# Compare sentence correction modes
Run the loading cell, the run-function cell, then the comparison cell. Every input runs first with `score="full string"`, then every input runs with `score="window"`. Each run uses a fresh solver and the same W, M, and timeout.

Each output shows completed full-sequence iterations, greedy steps, corrected text, and full-sentence cost. A non-improving step advances the window early; passes repeat until timeout. Global best updates occur every M * 10 steps.

Window mode scores up to 10 * W words total around the editable W-word window. Extra words are split across left and right, shifting at sentence edges. Full-string mode scores the entire sentence.


In [5]:
import json
import pickle
import importlib
import solvers
from wrapt_timeout_decorator import timeout

importlib.reload(solvers)
SentenceCorrector = solvers.SentenceCorrector

with open('data/conf_matrix.json') as f:
    conf_matrix = json.load(f)
with open('data/lm_model.pkl', 'rb') as f:
    model = pickle.load(f)
model.unk_prob = 1e-20
model.set_mode('spell_check')
with open('data/input.txt') as f:
    texts = [line.strip() for line in f]


In [6]:
def run(W=3, seconds=2, verbose=True, return_details=False):
    if not isinstance(W, int) or W < 1:
        raise ValueError('W must be a positive integer')
    # if not isinstance(k, int) or k < 2 or k % 2:
        # raise ValueError('k must be an even integer of at least 2')
    if seconds <= 0:
        raise ValueError('seconds must be positive')
    print("\n=== Window genetic: 10 * W context ===", flush=True)
    total_length = sum(len(text) for text in texts)
    if total_length == 0:
        raise ValueError('Input is empty')
    weighted_improvement = 0
    timed_out = 0
    sentence_costs = []

    @timeout(seconds)
    def solve(solver, text):
        solver.search(text)

    for text in texts:
        solver = SentenceCorrector(model, conf_matrix)
        solver.best_state = text
        initial_cost = model(text)
        did_timeout = False
        try:
            solve(solver, text)
        except TimeoutError:
            timed_out += 1
            did_timeout = True
        final_cost = model(solver.best_state)
        sentence_costs.append(final_cost)
        if verbose:
            # print(f'Greedy iterations: {solver.greedy_iterations}')
            # print(f'Beam iterations: {solver.beam_iterations}')
            # print(f'Steps: {solver.num_of_steps}')
            if did_timeout:
                print('Solver timed out; using best sentence found.')
            print(f'Input: {text}\nOutput: {solver.best_state}\nCost: {final_cost:.6f}\n')
        weighted_improvement += (initial_cost - final_cost) * len(text)

    improvement = weighted_improvement / total_length
    print(f'Scoring=window | W={W} | Weighted improvement: {improvement:.6f} | Timeouts: {timed_out}')
    if return_details:
        return improvement, sentence_costs
    return improvement

In [7]:
# Reload the window-only solver, then run every example.
importlib.reload(solvers)
SentenceCorrector = solvers.SentenceCorrectorCuti
W = 5

seconds = 2
window_score = run(W=W, seconds=seconds)



=== Window genetic: 10 * W context ===
Solver timed out; using best sentence found.
Input: he siit bass poshed for sale ov the independent television stations because the company payd too much the company said tte profit decline will de the reselt of an increase in reservec for inventory veluation the company caid president and chief executive officer mirris moliver had beun chief oberating officir as well
Output: he suit bass pushed for sale of the independent television stations because the company paid too much the company said the profit decline will be the result of an increase in reserves for inventory valuation the company said president and chief executive officer morris moliver had been chief operating officer as well
Cost: 57.030527

Solver timed out; using best sentence found.
Input: the dank ov finland hab never befori been allowed to iscue its own cds they said che world farm market suffers from curplucec because ef riseng production by exporting countries ha added three 